# 24 — Frozen spatiotemporal official-test evaluation

Evaluates the epoch-17 checkpoint once on the official test split using the validation-selected threshold. It performs no training and no threshold selection.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.8"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913
YEARS = tuple(range(2013, 2023))
VARIABLES = ("DBZ", "KDP", "RHOHV", "VEL", "WIDTH", "ZDR")
CHANNEL_ORDER = tuple(
    f"{variable}_sweep_{sweep}"
    for variable in VARIABLES
    for sweep in range(2)
)
FILE_BATCH_SIZE = 8
NUM_WORKERS = 12
SELECTED_THRESHOLD = 0.9195385575294495
EXPECTED_CHECKPOINT_EPOCH = 17

BACKUP_ROOT = Path("/content/drive/MyDrive/TorNet_Backup")
PACKAGE_PATH = BACKUP_ROOT / "packages" / f"tornet_detection-{PACKAGE_VERSION}-py3-none-any.whl"
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
NORMALIZATION_PATH = BACKUP_ROOT / "experiments" / "all_year_all6_baseline_v1" / "normalization.json"
EXPERIMENT_DIRECTORY = BACKUP_ROOT / "experiments" / "spatiotemporal_all_year_v1"
BEST_CHECKPOINT_PATH = EXPERIMENT_DIRECTORY / "best_model.pt"
OFFICIAL_TEST_METRICS_PATH = EXPERIMENT_DIRECTORY / "official_test_metrics.json"
EXTRACTION_ROOT = Path("/content/tornet_spatiotemporal_official_test")

for path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    NORMALIZATION_PATH,
    BEST_CHECKPOINT_PATH,
):
    if not path.exists():
        raise FileNotFoundError(path)

if OFFICIAL_TEST_METRICS_PATH.exists():
    raise FileExistsError(
        f"Official test was already evaluated: {OFFICIAL_TEST_METRICS_PATH}"
    )

print("checkpoint:", BEST_CHECKPOINT_PATH)
print("frozen threshold:", SELECTED_THRESHOLD)


checkpoint: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_all_year_v1/best_model.pt
frozen threshold: 0.9195385575294495


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
        "scikit-learn>=1.5",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.8-py3-none-any.whl'], returncode=0)

In [4]:
import datetime
import json
import random
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import (
    DataLoader,
    Dataset,
)

import tornado_detection
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
    read_spatiotemporal_netcdf_file,
)
from tornado_detection.models import (
    SpatiotemporalTornadoDetector,
)

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select an A100 GPU runtime and run all cells"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "The selected GPU does not support bfloat16"
    )


device = torch.device("cuda")

normalization = json.loads(
    NORMALIZATION_PATH.read_text()
)

expected_normalization = {
    "package_version": "0.1.7",
    "years": list(YEARS),
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": VALIDATION_SEED,
    "training_frame_count": 547_672,
    "training_file_count": 136_918,
    "training_positive_frame_count": 18_230,
    "variables": list(VARIABLES),
    "channel_order": list(CHANNEL_ORDER),
    "tensor_shape": [
        120,
        240,
        len(CHANNEL_ORDER),
    ],
}

mismatches = {
    key: {
        "expected": expected,
        "actual": normalization.get(key),
    }
    for key, expected
    in expected_normalization.items()
    if normalization.get(key) != expected
}

if mismatches:
    raise AssertionError(
        "Normalization provenance mismatch: "
        f"{mismatches}"
    )

means = np.asarray(
    normalization["means"],
    dtype=np.float32,
).reshape(
    len(VARIABLES),
    2,
).T
stds = np.asarray(
    normalization["standard_deviations"],
    dtype=np.float32,
).reshape(
    len(VARIABLES),
    2,
).T

assert means.shape == (2, 6)
assert stds.shape == (2, 6)
assert np.isfinite(means).all()
assert np.isfinite(stds).all()
assert (stds > 0).all()

canonical = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned = assign_model_splits(
    canonical,
    validation_fraction=VALIDATION_FRACTION,
    seed=VALIDATION_SEED,
)

test_rows = (
    assigned.loc[assigned["model_split"].eq("test")]
    .sort_values(["archive_member", "frame_index"])
)
assert len(test_rows) == 125_868
assert int(test_rows["frame_label"].sum()) == 3_909

if not test_rows.groupby("archive_member").size().eq(4).all():
    raise AssertionError(
        "Every official-test file must contain four frames"
    )

test_records = [
    (
        member,
        group["frame_label"].astype(np.uint8).to_numpy(),
    )
    for member, group
    in test_rows.groupby("archive_member", sort=True)
]
assert len(test_records) == 31_467

required_by_year = {
    year: set(
        test_rows.loc[
            test_rows["year"].eq(year),
            "archive_member",
        ].unique()
    )
    for year in YEARS
}

print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
print(
    "official-test files/frames:",
    len(test_records),
    len(test_rows),
)


GPU: NVIDIA A100-SXM4-40GB
torch: 2.11.0+cu128
official-test files/frames: 31467 125868


In [5]:
if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

staging = []

for year in YEARS:
    drive_archive = (
        BACKUP_ROOT
        / f"tornet_{year}.tar.gz"
    )
    local_archive = Path(
        f"/content/tornet_{year}.tar.gz"
    )

    if not drive_archive.is_file():
        raise FileNotFoundError(
            drive_archive
        )

    if local_archive.exists():
        local_archive.unlink()

    copy_started = (
        time.perf_counter()
    )

    shutil.copyfile(
        drive_archive,
        local_archive,
    )

    copy_seconds = (
        time.perf_counter()
        - copy_started
    )

    required = required_by_year[year]
    extracted = set()
    extract_started = (
        time.perf_counter()
    )

    with tarfile.open(
        local_archive,
        mode="r:gz",
    ) as archive:
        for member in archive:
            if (
                not member.isfile()
                or member.name
                not in required
            ):
                continue

            destination = (
                EXTRACTION_ROOT
                / member.name
            )
            destination.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            source_file = (
                archive.extractfile(
                    member
                )
            )

            if source_file is None:
                raise RuntimeError(
                    member.name
                )

            with (
                source_file,
                destination.open("wb")
                as output_file,
            ):
                shutil.copyfileobj(
                    source_file,
                    output_file,
                    length=1024 * 1024,
                )

            extracted.add(
                member.name
            )

    extraction_seconds = (
        time.perf_counter()
        - extract_started
    )
    missing = required - extracted

    if missing:
        raise RuntimeError(
            f"Year {year} missing: "
            f"{sorted(missing)[:10]}"
        )

    local_archive.unlink()

    staging.append(
        {
            "year": year,
            "file_count": len(
                extracted
            ),
            "copy_seconds": (
                copy_seconds
            ),
            "extraction_seconds": (
                extraction_seconds
            ),
        }
    )

    print(
        f"year={year} "
        f"staged_files="
        f"{len(extracted):,} "
        f"copy={copy_seconds:.1f}s "
        f"extract="
        f"{extraction_seconds:.1f}s"
    )

print(
    "total staged files:",
    sum(
        row["file_count"]
        for row in staging
    ),
)

staged_bytes = sum(
    path.stat().st_size
    for path
    in EXTRACTION_ROOT.rglob("*.nc")
)

print(
    "staged GiB:",
    round(
        staged_bytes / 1024**3,
        3,
    ),
)

year=2013 staged_files=573 copy=86.6s extract=9.3s
year=2014 staged_files=2,546 copy=388.3s extract=43.8s
year=2015 staged_files=3,902 copy=294.6s extract=53.7s
year=2016 staged_files=2,951 copy=508.1s extract=48.9s
year=2017 staged_files=3,145 copy=285.7s extract=43.9s
year=2018 staged_files=2,518 copy=362.1s extract=37.7s
year=2019 staged_files=4,031 copy=439.0s extract=56.3s
year=2020 staged_files=4,756 copy=356.6s extract=51.0s
year=2021 staged_files=4,268 copy=410.8s extract=55.1s
year=2022 staged_files=2,777 copy=407.6s extract=58.2s
total staged files: 31467
staged GiB: 23.93


In [6]:
class SequenceDataset(Dataset):
    def __init__(
        self,
        file_records,
        root,
        channel_means,
        channel_stds,
    ):
        self.records = file_records
        self.root = root
        self.means = channel_means.reshape(
            1,
            2,
            6,
            1,
            1,
        )
        self.stds = channel_stds.reshape(
            1,
            2,
            6,
            1,
            1,
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        (
            member,
            expected_labels,
        ) = self.records[index]

        sequence = (
            read_spatiotemporal_netcdf_file(
                self.root / member,
                variables=VARIABLES,
            )
        )

        np.testing.assert_array_equal(
            sequence.labels,
            expected_labels,
        )

        normalized = (
            (
                sequence.values
                - self.means
            )
            / self.stds
        ).astype(
            np.float32,
            copy=False,
        )

        return (
            torch.from_numpy(normalized),
            torch.from_numpy(
                sequence.finite_mask
            ),
            torch.from_numpy(
                sequence.range_folded_mask
            ),
            torch.from_numpy(
                sequence.coordinates
            ),
            torch.from_numpy(
                sequence.labels.astype(
                    np.float32
                )
            ),
        )



test_dataset = SequenceDataset(
    test_records,
    EXTRACTION_ROOT,
    means,
    stds,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=FILE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

model = SpatiotemporalTornadoDetector().to(device)
checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

if checkpoint.get("artifact_kind") != "spatiotemporal_all_year_best_model":
    raise AssertionError("Unexpected checkpoint artifact")

if int(checkpoint["epoch"]) != EXPECTED_CHECKPOINT_EPOCH:
    raise AssertionError(
        f"Expected epoch {EXPECTED_CHECKPOINT_EPOCH}; "
        f"found {checkpoint['epoch']}"
    )

model.load_state_dict(checkpoint["model_state_dict"])

train_positive_count = 18_230
train_negative_count = 547_672 - train_positive_count
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        train_negative_count / train_positive_count,
        device=device,
    )
)

print("checkpoint epoch:", checkpoint["epoch"])
print(
    "parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}",
)


checkpoint epoch: 17
parameters: 2,011,201


In [7]:
def move_batch_to_device(batch):
    (
        values,
        finite_mask,
        range_folded_mask,
        coordinates,
        targets,
    ) = batch

    return (
        values.to(
            device,
            non_blocking=True,
        ),
        finite_mask.to(
            device,
            non_blocking=True,
        ),
        range_folded_mask.to(
            device,
            non_blocking=True,
        ),
        coordinates.to(
            device,
            non_blocking=True,
        ),
        targets.to(
            device,
            non_blocking=True,
        ),
    )


def evaluate(loader):
    model.eval()

    labels = []
    probabilities = []
    loss_sum = 0.0
    count = 0

    with torch.no_grad():
        for batch in loader:
            (
                values,
                finite_mask,
                range_folded_mask,
                coordinates,
                targets,
            ) = move_batch_to_device(
                batch
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                logits = model(
                    values,
                    finite_mask,
                    range_folded_mask,
                    coordinates,
                )
                loss = criterion(
                    logits,
                    targets,
                )

            size = int(
                targets.numel()
            )
            loss_sum += (
                float(loss.cpu())
                * size
            )
            count += size

            labels.extend(
                targets.cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )
            probabilities.extend(
                logits.sigmoid()
                .float()
                .cpu()
                .numpy()
                .reshape(-1)
                .tolist()
            )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    return (
        loss_sum / count,
        float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        labels,
        probabilities,
    )




In [8]:
test_loss, test_pr_auc, test_roc_auc, labels, probabilities = evaluate(
    test_loader
)
predictions = (
    probabilities >= SELECTED_THRESHOLD
).astype(np.int64)

precision = float(
    precision_score(labels, predictions, zero_division=0)
)
recall = float(
    recall_score(labels, predictions, zero_division=0)
)
f1 = float(
    2 * precision * recall
    / max(precision + recall, 1e-12)
)

tn, fp, fn, tp = [
    int(value)
    for value in confusion_matrix(
        labels,
        predictions,
        labels=[0, 1],
    ).ravel()
]

frame_results = test_rows[
    ["year", "frame_id", "frame_label"]
].copy()
frame_results["probability"] = probabilities
frame_results["prediction"] = predictions

by_year = []

for year, rows in frame_results.groupby("year", sort=True):
    year_labels = rows["frame_label"].to_numpy(dtype=np.int64)
    year_probabilities = rows["probability"].to_numpy(dtype=np.float64)
    year_predictions = rows["prediction"].to_numpy(dtype=np.int64)

    year_precision = float(
        precision_score(
            year_labels,
            year_predictions,
            zero_division=0,
        )
    )
    year_recall = float(
        recall_score(
            year_labels,
            year_predictions,
            zero_division=0,
        )
    )

    by_year.append(
        {
            "year": int(year),
            "frame_count": int(len(rows)),
            "positive_count": int(year_labels.sum()),
            "pr_auc": float(
                average_precision_score(
                    year_labels,
                    year_probabilities,
                )
            ),
            "roc_auc": float(
                roc_auc_score(
                    year_labels,
                    year_probabilities,
                )
            ),
            "precision": year_precision,
            "recall": year_recall,
            "f1": float(
                2 * year_precision * year_recall
                / max(
                    year_precision + year_recall,
                    1e-12,
                )
            ),
        }
    )

metrics = {
    "artifact_kind": "spatiotemporal_all_year_official_test_metrics",
    "created_at_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "package_version": PACKAGE_VERSION,
    "model_class": "SpatiotemporalTornadoDetector",
    "variables": list(VARIABLES),
    "channel_order": list(CHANNEL_ORDER),
    "sequence_shape": [4, 2, 6, 120, 240],
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "selected_threshold": SELECTED_THRESHOLD,
    "threshold_selection": "frozen maximum-validation-F1 threshold",
    "test_frame_count": int(len(labels)),
    "test_positive_count": int(labels.sum()),
    "test_loss": test_loss,
    "test_pr_auc": test_pr_auc,
    "test_roc_auc": test_roc_auc,
    "test_precision": precision,
    "test_recall": recall,
    "test_f1": f1,
    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp,
    "by_year": by_year,
    "staging": staging,
}

OFFICIAL_TEST_METRICS_PATH.write_text(
    json.dumps(metrics, indent=2, sort_keys=True) + "\n"
)

print(json.dumps(metrics, indent=2, sort_keys=True))
print("wrote:", OFFICIAL_TEST_METRICS_PATH)


{
  "artifact_kind": "spatiotemporal_all_year_official_test_metrics",
  "by_year": [
    {
      "f1": 0.7418397626112759,
      "frame_count": 2292,
      "positive_count": 157,
      "pr_auc": 0.8296034235447302,
      "precision": 0.6944444444444444,
      "recall": 0.7961783439490446,
      "roc_auc": 0.9740136338549203,
      "year": 2013
    },
    {
      "f1": 0.5983379501385041,
      "frame_count": 10184,
      "positive_count": 792,
      "pr_auc": 0.6353663427147168,
      "precision": 0.6625766871165644,
      "recall": 0.5454545454545454,
      "roc_auc": 0.900812923205651,
      "year": 2014
    },
    {
      "f1": 0.5267924528301887,
      "frame_count": 15608,
      "positive_count": 670,
      "pr_auc": 0.5424762556344318,
      "precision": 0.5328244274809161,
      "recall": 0.5208955223880597,
      "roc_auc": 0.9242355467274687,
      "year": 2015
    },
    {
      "f1": 0.3774733637747336,
      "frame_count": 11804,
      "positive_count": 333,
      "pr_auc":

In [9]:
shutil.rmtree(EXTRACTION_ROOT)

assert not EXTRACTION_ROOT.exists()
assert BEST_CHECKPOINT_PATH.is_file()
assert OFFICIAL_TEST_METRICS_PATH.is_file()

print("Removed all Colab-local test data")
print("Preserved:", OFFICIAL_TEST_METRICS_PATH)


Removed all Colab-local test data
Preserved: /content/drive/MyDrive/TorNet_Backup/experiments/spatiotemporal_all_year_v1/official_test_metrics.json
